# Fine-Tune Gemma 4 E2B-IT (QLoRA) — Safe Version
Small model (~2.3B effective params). Fits easily on a single T4.
Final GGUF: ~1.5 GB.

In [ ]:
!pip uninstall -y torchao torchvision

In [1]:
!pip install -q -U git+https://github.com/huggingface/transformers.git peft bitsandbytes accelerate datasets huggingface_hub


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 12.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 105.2 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.25 requires torchvision, which is not installed.


In [2]:
import os, sys, torch, gc
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
BASE_DIR = '/kaggle/working'
MODEL_OUT = os.path.join(BASE_DIR, 'gemma_lora_output')
MERGED_DIR = os.path.join(BASE_DIR, 'gemma_merged_fp16')
os.makedirs(MODEL_OUT, exist_ok=True); os.makedirs(MERGED_DIR, exist_ok=True)
model_id = 'google/gemma-4-E2B-it'
print(f'GPUs: {torch.cuda.device_count()}')


GPUs: 2


In [ ]:
from huggingface_hub import login
os.environ['WANDB_DISABLED'] = 'true'
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    print('Logged in via Kaggle Secrets')
except Exception:
    if 'HF_TOKEN' in os.environ: login(token=os.environ['HF_TOKEN'])
    else: print('WARNING: No HF_TOKEN found!')


In [ ]:
from datasets import load_dataset
ds = load_dataset('ajibawa-2023/Children-Stories-Collection', split='train', cache_dir=os.path.join(BASE_DIR, 'hf_cache'))
sample_size = 10000
ds_sample = ds.shuffle(seed=42).select(range(min(sample_size, len(ds))))
PREFIXES = [
    'Level: Age3-4 — Simple words.', 'Level: Age5-6 — Short sentences.',
    'Level: Age7-8 — Moderate vocabulary.', 'Level: Age9-10 — Longer sentences.',
    'Level: Age11-12 — Richer vocabulary.'
]
def format_gemma(ex, idx):
    pfx = PREFIXES[idx % len(PREFIXES)]
    return {'text': f'<start_of_turn>user\n{pfx}\n{ex.get("prompt","")}<end_of_turn>\n<start_of_turn>model\n{ex.get("text","")}<end_of_turn>\n'}
ds_fmt = ds_sample.map(format_gemma, with_indices=True, remove_columns=ds_sample.column_names)
print(f'Formatted {len(ds_fmt)} samples')


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map='auto')
print('Model loaded successfully!')


In [ ]:
from peft import LoraConfig, get_peft_model
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
for n, p in model.named_parameters():
    p.requires_grad = False
    if p.ndim == 1 and 'norm' in n.lower(): p.data = p.data.to(torch.float32)
try:
    from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
    ct = 0
    for n, m in list(model.named_modules()):
        if isinstance(m, Gemma4ClippableLinear):
            parts = n.split('.')
            setattr(model.get_submodule('.'.join(parts[:-1])), parts[-1], m.linear)
            ct += 1
    print(f'Unwrapped {ct} ClippableLinear modules')
except (ImportError, AttributeError) as e:
    print(f'Unwrap skipped: {e}')
lora_config = LoraConfig(r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
MAX_SEQ_LEN = 256
def tok_fn(ex):
    o = tokenizer(ex['text'], truncation=True, max_length=MAX_SEQ_LEN, padding='max_length')
    o['labels'] = o['input_ids'].copy()
    return o
ds_tok = ds_fmt.map(tok_fn, batched=True, remove_columns=['text'])
sp = ds_tok.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = sp['train'], sp['test']
print(f'Train: {len(train_ds)}, Eval: {len(eval_ds)}')


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
args = TrainingArguments(output_dir=MODEL_OUT, per_device_train_batch_size=1, gradient_accumulation_steps=16,
    optim='paged_adamw_8bit', save_steps=200, save_total_limit=2, logging_steps=20,
    learning_rate=2e-4, max_grad_norm=0.3, num_train_epochs=1, warmup_steps=10,
    lr_scheduler_type='cosine', fp16=True, gradient_checkpointing=True,
    eval_strategy='no')  # Eval disabled — 256k vocab logits cause OOM during eval
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
print('Training with batch_size=1, grad_accum=16, gradient_checkpointing=True')
# Resume from checkpoint if one exists (e.g. after a crash)
import glob
ckpts = sorted(glob.glob(os.path.join(MODEL_OUT, 'checkpoint-*')))
resume = ckpts[-1] if ckpts else None
if resume: print(f'Resuming from {resume}')
trainer.train(resume_from_checkpoint=resume)
trainer.model.save_pretrained(MODEL_OUT); tokenizer.save_pretrained(MODEL_OUT)


In [ ]:
print('Freeing GPU memory...')
del model, trainer
torch.cuda.empty_cache()
gc.collect()


In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
print('Loading base model in fp16 on CPU for merge...')
base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map='cpu')
try:
    from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
    for n, m in list(base.named_modules()):
        if isinstance(m, Gemma4ClippableLinear):
            parts = n.split('.')
            setattr(base.get_submodule('.'.join(parts[:-1])), parts[-1], m.linear)
except Exception as e: print(e)
merged = PeftModel.from_pretrained(base, MODEL_OUT).merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved to {MERGED_DIR}')
del base, merged; gc.collect()
print('Aggressively cleaning disk space...')
os.system('rm -rf ~/.cache/huggingface/hub')
os.system(f'rm -rf {MODEL_OUT}')


In [ ]:
%%bash
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp && cmake -B build && cmake --build build --config Release -j 4


In [ ]:
%%bash
pip install ./llama.cpp/gguf-py
python llama.cpp/convert_hf_to_gguf.py /kaggle/working/gemma_merged_fp16 --outfile /kaggle/working/ft-gemma-e2b-fp16.gguf --outtype f16
rm -rf /kaggle/working/gemma_merged_fp16
./llama.cpp/build/bin/llama-quantize /kaggle/working/ft-gemma-e2b-fp16.gguf /kaggle/working/ft-gemma-e2b-Q4_K_M.gguf Q4_K_M
rm -f /kaggle/working/ft-gemma-e2b-fp16.gguf
ls -lh /kaggle/working/ft-gemma-e2b-Q4_K_M.gguf


In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    login(token=user_secrets.get_secret('HF_TOKEN'))
except Exception as e:
    print('Warning: HF_TOKEN not found in secrets. If upload fails, login manually.', e)
api = HfApi()
GGUF = '/kaggle/working/ft-gemma-e2b-Q4_K_M.gguf'
api.upload_file(path_or_fileobj=GGUF, path_in_repo='finetuned-gemma-4-e2b-it-Q4_K_M.gguf', repo_id='khedim/NLP-MINI-PROJECT',
    commit_message='Upload fine-tuned Gemma 4 E2B Q4_K_M GGUF')
print('Upload Complete!')


In [3]:
import os
import gc
import torch
from huggingface_hub import login, HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from kaggle_secrets import UserSecretsClient

# ==========================================
# 1. SETUP & AUTHENTICATION
# ==========================================
try:
    user_secrets = UserSecretsClient()
    login(token=user_secrets.get_secret("HF_TOKEN"))
except Exception as e:
    print("Warning: HF_TOKEN not found in secrets. If upload fails, login manually.", e)

BASE_DIR = '/kaggle/working'
MODEL_OUT = os.path.join(BASE_DIR, 'gemma_lora_output')
MERGED_DIR = os.path.join(BASE_DIR, 'gemma_merged_fp16')
model_id = 'google/gemma-4-2b-it' # Base model ID

# Check if we survived!
if not os.path.exists(MODEL_OUT):
    print("❌ FATAL: The training output folder 'gemma_lora_output' is completely gone. You will need to restart the notebook from scratch.")
    exit(1)

# ==========================================
# 2. MERGE LORA ADAPTER INTO BASE MODEL
# ==========================================
print('\n--- MERGING LORA WEIGHTS ---')
tokenizer = AutoTokenizer.from_pretrained(model_id)
base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map='cpu')

try:
    from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
    for n, m in list(base.named_modules()):
        if isinstance(m, Gemma4ClippableLinear):
            parts = n.split('.')
            setattr(base.get_submodule('.'.join(parts[:-1])), parts[-1], m.linear)
except Exception as e:
    print("Unwrapping error (can be ignored if already unwrapped):", e)

merged = PeftModel.from_pretrained(base, MODEL_OUT).merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model successfully saved to {MERGED_DIR}')

del base, merged
gc.collect()

# ==========================================
# 3. AGGRESSIVELY CLEAR DISK SPACE
# ==========================================
print('\n--- CLEANING DISK FOR CONVERSION ---')
# The merge was successful, we no longer need the raw checkpoints or the base model cache!
os.system('rm -rf ~/.cache/huggingface/hub')
os.system(f'rm -rf {MODEL_OUT}')
os.system(f'rm -f {BASE_DIR}/ft-gemma-e2b-fp16.gguf') # delete any failed gguf attempt
os.system('df -h /kaggle/working')

# ==========================================
# 4. CONVERT TO GGUF & QUANTIZE
# ==========================================
print('\n--- CONVERTING TO GGUF AND QUANTIZING ---')
os.system('pip install ./llama.cpp/gguf-py')

# Convert FP16
print("Converting HF to GGUF (FP16)...")
os.system(f'python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {BASE_DIR}/ft-gemma-e2b-fp16.gguf --outtype f16')

# Delete merged safetensors immediately!
os.system(f'rm -rf {MERGED_DIR}')

# Quantize to Q4_K_M
print("Quantizing to Q4_K_M...")
os.system(f'./llama.cpp/build/bin/llama-quantize {BASE_DIR}/ft-gemma-e2b-fp16.gguf {BASE_DIR}/ft-gemma-e2b-Q4_K_M.gguf Q4_K_M')

# Delete FP16 GGUF
os.system(f'rm -f {BASE_DIR}/ft-gemma-e2b-fp16.gguf')

print('\n--- FINAL FILE ---')
os.system(f'ls -lh {BASE_DIR}/ft-gemma-e2b-Q4_K_M.gguf')

# ==========================================
# 5. UPLOAD TO HUGGING FACE
# ==========================================
print('\n--- UPLOADING TO HUGGING FACE ---')
api = HfApi()
gguf_path = os.path.join(BASE_DIR, 'ft-gemma-e2b-Q4_K_M.gguf')

try:
    api.upload_file(
        path_or_fileobj=gguf_path, 
        path_in_repo='finetuned-gemma-4-e2b-it-Q4_K_M.gguf', 
        repo_id='khedim/NLP-MINI-PROJECT',
        commit_message='Upload fine-tuned Gemma 4 E2B Q4_K_M GGUF'
    )
    print('\n🎉 UPLOAD COMPLETE! THE MIRACLE HAPPENED! 🎉')
except Exception as e:
    print("\n❌ Upload failed:", e)



--- MERGING LORA WEIGHTS ---


OSError: google/gemma-4-2b-it is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`